In [4]:



import pandas as pd
df = pd.read_csv("logistics_dataset.csv")
# df.head()
# df.info()
# df.describe()
# df.isnull().sum()
# df.duplicated().sum()
df.isnull().sum()
# df

Order_ID                      0
Order_Date                    0
Delivery_Date                 3
Promised_Date                 0
Product_Category              0
Quantity                      4
Warehouse                     0
Shipping_Mode                 0
Distance_KM                   7
Vehicle_Type                  0
Delivery_Partner              5
Warehouse_Processing_Hours    0
Delivery_Cost_INR             0
Delivery_Status               0
dtype: int64

In [15]:
df = df.drop_duplicates()
df


,Order_ID,Order_Date,Delivery_Date,Promised_Date,Product_Category,Quantity,Warehouse,Shipping_Mode,Distance_KM,Vehicle_Type,Delivery_Partner,Warehouse_Processing_Hours,Delivery_Cost_INR,Delivery_Status
0,1,2026-03-23,2026-03-25,2026-03-25,Personal Care,5.0,Gurgaon-WH,Standard,5.1,Van,Partner A,6,223.92,On Time
1,2,2026-01-04,2026-01-04,2026-01-04,Personal Care,9.0,Delhi-WH,Same Day,20.9,Bike,Partner B,6,378.63,On Time
2,3,2026-03-17,2026-03-20,2026-03-21,Clothing,1.0,Ghaziabad-WH,Standard,69.4,Truck,Partner C,12,536.53,On Time
3,4,2026-02-18,2026-02-18,2026-02-18,Personal Care,6.0,Noida-WH,Same Day,32.4,Van,Partner D,6,438.79,On Time
4,5,2026-02-18,2026-02-18,2026-02-18,Electronics,11.0,Noida-WH,Same Day,48.3,Bike,Partner B,6,605.48,On Time
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
245,246,2026-04-17,2026-04-18,2026-04-17,Clothing,7.0,Ghaziabad-WH,Same Day,70.2,Van,Partner C,6,664.72,Late
246,247,2026-04-02,2026-04-07,2026-04-05,Personal Care,2.0,Ghaziabad-WH,Express,NaN,Bike,Partner D,24,629.61,Late
247,248,2026-04-22,2026-04-24,2026-04-22,Home & Kitchen,6.0,Delhi-WH,Same Day,47.6,Van,Partner B,24,564.42,Late
248,249,2026-01-06,2026-01-10,2026-01-09,Home & Kitchen,6.0,Delhi-WH,Express,82.3,Van,Partner A,24,764.44,Late


In [5]:
#Data conversion
df["col"] =  df["Promised_Date"] < df["Delivery_Date"]
print(df["col"]) 
df["Order_Date"] = pd.to_datetime(df["Order_Date"])
df["Promised_Date"] = pd.to_datetime(df["Promised_Date"])

#Data standarizing
df["Shipping_Mode"] = (
    df["Shipping_Mode"]
    .astype("string")
    .str.strip()
    .str.title()
)
df["Shipping_Mode"]

0      False
1      False
2      False
3      False
4      False
       ...  
248     True
249    False
250    False
251    False
252     True
Name: col, Length: 253, dtype: bool


0      Standard
1      Same Day
2      Standard
3      Same Day
4      Same Day
         ...   
248     Express
249     Express
250    Same Day
251    Standard
252    Standard
Name: Shipping_Mode, Length: 253, dtype: string

In [6]:
#df.isnull().sum()        // for finding how much null values in each columns
for col in ["Distance_KM", "Quantity", "Delivery_Cost_INR"]:
  df[col] = df[col].fillna(df[col].median())
    
df["Delivery_Partner"] = df["Delivery_Partner"].fillna("unknown")   
# df["Delivery_Date"] = df["Delivery_Date"].fillna("")
df.loc[df["Delivery_Cost_INR"] < 0, "Delivery_Cost_INR"] = pd.NA

df["Delivery_Cost_INR"] = df["Delivery_Cost_INR"].fillna(
    df["Delivery_Cost_INR"].median())




In [7]:
df.isnull().sum()


Order_ID                      0
Order_Date                    0
Delivery_Date                 3
Promised_Date                 0
Product_Category              0
Quantity                      0
Warehouse                     0
Shipping_Mode                 0
Distance_KM                   0
Vehicle_Type                  0
Delivery_Partner              0
Warehouse_Processing_Hours    0
Delivery_Cost_INR             0
Delivery_Status               0
col                           0
dtype: int64

In [14]:

# Interquartile range 
Q1 = df["Distance_KM"].quantile(0.25)
Q3 = df["Distance_KM"].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers = df[
    (df["Distance_KM"] < lower) | 
    (df["Distance_KM"] > upper)
]
print(IQR)
print(lower)
print(upper)
print(outliers)
for col in ["Quantity", "Distance_KM", "Delivery_Cost_INR"]:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    df[col] = df[col].clip(lower=lower, upper=upper)

40.099999999999994
-34.64999999999999
125.74999999999999
Empty DataFrame
Columns: [Order_ID, Order_Date, Delivery_Date, Promised_Date, Product_Category, Quantity, Warehouse, Shipping_Mode, Distance_KM, Vehicle_Type, Delivery_Partner, Warehouse_Processing_Hours, Delivery_Cost_INR, Delivery_Status, col]
Index: []


In [25]:
df["Order_Date"] = pd.to_datetime(df["Order_Date"]) # converted above
df["Promised_Date"] = pd.to_datetime(df["Promised_Date"]) #converted above
df["Delivery_Date"] = pd.to_datetime(df["Delivery_Date"]) # converted now
#feature engineering
df["Delivery_Time_Days"] = (
    df["Delivery_Date"] - df["Order_Date"]).dt.days
print(df["Delivery_Time_Days"])
df["Late_Flag"] = (
    df["Delivery_Date"].notna()
    & df["Promised_Date"].notna()
    & (df["Delivery_Date"] > df["Promised_Date"])
)

0      2.0
1      0.0
2      3.0
3      0.0
4      0.0
      ... 
248    4.0
249    1.0
250    0.0
251    3.0
252    4.0
Name: Delivery_Time_Days, Length: 253, dtype: float64


In [28]:
pip install scikit-learn  #Supervised learning  important library


   ---------------------------------------- 0.0/8.9 MB ? eta -:--:--
   ----------------- ---------------------- 3.9/8.9 MB 19.6 MB/s eta 0:00:01
   ------------------------ --------------- 5.5/8.9 MB 12.9 MB/s eta 0:00:01
   ---------------------------------------- 8.9/8.9 MB 14.5 MB/s  0:00:00
   ---------------------------------------- 0.0/41.3 MB ? eta -:--:--
   -- ------------------------------------- 2.6/41.3 MB 12.6 MB/s eta 0:00:04
   --- ------------------------------------ 3.7/41.3 MB 8.4 MB/s eta 0:00:05
   ------- -------------------------------- 8.1/41.3 MB 12.6 MB/s eta 0:00:03
   ---------- ----------------------------- 10.5/41.3 MB 12.3 MB/s eta 0:00:03
   ------------ --------------------------- 12.8/41.3 MB 12.0 MB/s eta 0:00:03
   -------------- ------------------------- 15.5/41.3 MB 12.2 MB/s eta 0:00:03
   ----------------- ---------------------- 18.1/41.3 MB 12.3 MB/s eta 0:00:02
   ------------------- -------------------- 20.4/41.3 MB 12.1 MB/s eta 0:00:02
   --

In [29]:
from sklearn.preprocessing import MinMaxScaler        # don"t know which library function to use so little bit of ai is used here
scaler = MinMaxScaler()

numeric_cols = [ "Quantity","Distance_KM","Delivery_Cost_INR"]

df[[col + "_Normalized" for col in numeric_cols]] = (
    scaler.fit_transform(df[numeric_cols]))

In [33]:
#filtered or final data  
df.describe()
df.isnull().sum()

Order_ID                        0
Order_Date                      0
Delivery_Date                   3
Promised_Date                   0
Product_Category                0
Quantity                        0
Warehouse                       0
Shipping_Mode                   0
Distance_KM                     0
Vehicle_Type                    0
Delivery_Partner                0
Warehouse_Processing_Hours      0
Delivery_Cost_INR               0
Delivery_Status                 0
col                             0
Delivery_Time_Days              3
Quantity_Normalized             0
Distance_KM_Normalized          0
Delivery_Cost_INR_Normalized    0
dtype: int64